# Production Agent Harness

An agent harness is the application layer around a model that manages state, tools, control flow, safety, and observability.


## What to learn

- State records the task, evidence, decisions, and current status.
- The control loop decides whether to call a tool, ask a person, retry, or finish.
- Tool adapters validate inputs and isolate external side effects.
- Checkpoints and traces make failures recoverable and explainable.

## Decision rule

Begin with one explicit loop and typed state. Add planners, evaluators, graphs, or multiple agents only when a measured failure requires them. Make budgets, permissions, termination, and recovery visible in code.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
# ALTERNATIVE FRAMEWORK EXAMPLE: Agno agent prepared for AgentOS
# Import the dependencies used by this example.
import os

if not os.getenv("OPENAI_API_KEY"):
    print("Skipped: set OPENAI_API_KEY to run this Agno example.")
else:
    from agno.agent import Agent

    # Define the data shape and small operations before running them.
    def lookup_runbook(service: str) -> str:
        """Return the first recovery step for a known service."""
        runbooks = {"checkout": "Check payment-provider health and queue depth."}
        return runbooks.get(service, "Escalate to the owning team.")

    # Configure the framework object; this line prepares it but may not execute it yet.
    agent = Agent(
        id="operations-assistant",
        name="Operations assistant",
        model="openai:gpt-5.6-sol",
        instructions="Use the runbook tool. State evidence before recommending action.",
        tools=[lookup_runbook],
        markdown=True,
    )

    # Agno runs the same Agent object directly during development; AgentOS can
    # later expose it through managed run/session APIs.
    agent.print_response("Checkout errors increased. What should I inspect first?")


In [ ]:
# Import the dependencies used by this example.
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import InMemorySaver

reviewer = {
    "name": "reviewer",
    "description": "Check an implementation against acceptance criteria.",
    "system_prompt": "Return defects and evidence only. Do not rewrite the implementation.",
    "tools": [],
}

# Configure the framework object; this line prepares it but may not execute it yet.
agent = create_deep_agent(
    model="openai:gpt-5.6-sol",
    system_prompt="Plan the task, implement it, delegate review, then fix verified defects.",
    subagents=[reviewer],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "harness-lab-1"}}
# Execute the configured model or workflow with the input below.
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Draft and review a rollout checklist."}]},
    config=config,
)
result["messages"][-1].content


In [ ]:
"""A miniature typed-state-graph harness with checkpointing and an
evaluator gate. Pure Python — the structure mirrors LangGraph exactly:
typed state, nodes returning partial updates, a reducer, a checkpointer
keyed by thread_id, and a bounded generate->evaluate loop.
"""
# Import the dependencies used by this example.
import json

CHECKPOINTS = {}  # thread_id -> list of state snapshots (prod: Postgres)


# Define the data shape and small operations before running them.
def checkpoint(thread_id, node, state):
    CHECKPOINTS.setdefault(thread_id, []).append(
        {"node": node, "state": json.loads(json.dumps(state))})


def merge(state, update):
    """Reducer: 'feedback' appends (safe under fan-in); others overwrite."""
    out = {**state}
    for k, v in update.items():
        out[k] = out.get(k, []) + v if k == "feedback" else v
    return out


# --- nodes: each returns a PARTIAL state update ---------------------------
def planner(state):
    return {"plan": ["draft summary", "cite evidence"]}


def generator(state):
    # Improves with accumulated feedback (simulates revision)
    quality = 0.5 + 0.25 * len(state["feedback"])
    return {"draft": f"summary v{state['iteration'] + 1}", "quality": quality,
            "iteration": state["iteration"] + 1}


def evaluator(state):
    """Hybrid gate: deterministic check + rubric score with cited defect."""
    if state["quality"] >= 0.9:
        return {"verdict": "PASS"}
    return {"verdict": "REVISE",
            "feedback": [f"v{state['iteration']}: evidence citations missing"]}


def run(thread_id, max_iterations=3):
    state = {"iteration": 0, "feedback": [], "verdict": None}
    state = merge(state, planner(state))
    checkpoint(thread_id, "planner", state)
    while state["iteration"] < max_iterations:
        state = merge(state, generator(state))
        checkpoint(thread_id, "generator", state)
        state = merge(state, evaluator(state))
        checkpoint(thread_id, "evaluator", state)
        if state["verdict"] == "PASS":
            break
    return state


final = run("thread-42")
print(f"verdict={final['verdict']} after {final['iteration']} iterations")
print("feedback history:", final["feedback"])
print(f"checkpoints persisted: {len(CHECKPOINTS['thread-42'])} "
      "(any of these can be resumed or forked)")
# time-travel: inspect state as it was after the first evaluation
print("state after first gate:", CHECKPOINTS["thread-42"][2]["state"]["verdict"])


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
